In [0]:
# Load and standardize the core dataset
from pyspark.sql import functions as F
from pyspark.sql.window import Window

events = spark.table("silver.events")

# Standardize time columns once
events = (
    events
    .withColumn("event_date", F.to_date("event_ts"))
    .withColumn("event_hour", F.hour("event_ts"))
    .withColumn("dow", F.dayofweek("event_ts"))                 # 1=Sun ... 7=Sat
    .withColumn("is_weekend", F.col("dow").isin([1,7]).cast("int"))
)

events.select("event_ts","event_date","event_hour","dow","is_weekend").show(5, truncate=False)


In [0]:
# Descriptive statistics that matter (not just describe(price))
## - Price stats for purchases only
price_stats = (
    events
    .filter(F.col("event_type") == "purchase")
    .select("price")
    .summary("count","min","25%","50%","75%","max","mean","stddev")
)

price_stats.show(truncate=False)

In [0]:
## - Event mix by type (overall + weekend split)
event_mix = (
    events
    .groupBy("is_weekend", "event_type")
    .count()
    .orderBy("is_weekend", F.desc("count"))
)
event_mix.show(truncate=False)

In [0]:
## - Daily revenue + conversion (foundation for stats)
daily = (
    events
    .groupBy("event_date", "is_weekend")
    .agg(
        F.sum(F.when(F.col("event_type")=="purchase", F.col("price")).otherwise(F.lit(0))).alias("revenue"),
        F.sum(F.when(F.col("event_type")=="view", 1).otherwise(0)).alias("views"),
        F.sum(F.when(F.col("event_type")=="purchase", 1).otherwise(0)).alias("purchases"),
        F.approx_count_distinct("user_id").alias("active_users"),
        F.approx_count_distinct(F.when(F.col("event_type")=="purchase", F.col("user_id"))).alias("purchasing_users"),
    )
    .withColumn("conversion_rate", F.when(F.col("views")==0, F.lit(0.0))
                .otherwise(F.col("purchases")/F.col("views")))
)

daily.orderBy("event_date").show(10, truncate=False)


In [0]:
# Reusable gold table:
(daily.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("gold.daily_kpis"))

In [0]:
# Hypothesis test: weekday vs weekend conversion (properly)
## - Preparing samples
weekend_rates = spark.table("gold.daily_kpis").select("is_weekend","conversion_rate")

wk = weekend_rates.filter("is_weekend = 0").select("conversion_rate")
we = weekend_rates.filter("is_weekend = 1").select("conversion_rate")

In [0]:
## - Two-sample t-test in Spark (no external libs required)
def agg_stats(df):
    r = df.agg(
        F.count("*").alias("n"),
        F.avg("conversion_rate").alias("mean"),
        F.var_samp("conversion_rate").alias("var")
    ).first()
    return r["n"], r["mean"], r["var"]

n1, m1, v1 = agg_stats(wk)
n2, m2, v2 = agg_stats(we)

t = (m1 - m2) / ((v1/n1 + v2/n2) ** 0.5) if (v1/n1 + v2/n2) > 0 else None

print("weekday n/mean/var:", n1, m1, v1)
print("weekend n/mean/var:", n2, m2, v2)
print("t-statistic:", t)

In [0]:
## - Adding effect size (Cohen’s d) so this isn’t “p-value theatre”:
import math
sp = math.sqrt(((n1-1)*v1 + (n2-1)*v2) / (n1+n2-2)) if (n1+n2-2) > 0 else None
d = (m1 - m2)/sp if sp and sp > 0 else None
print("Cohen's d:", d)

In [0]:
## - Persist the result as a gold table so it shows up in your catalog:
res = spark.createDataFrame([{
    "metric": "conversion_rate_daily",
    "weekday_mean": float(m1),
    "weekend_mean": float(m2),
    "weekday_n": int(n1),
    "weekend_n": int(n2),
    "t_stat": float(t) if t is not None else None,
    "cohens_d": float(d) if d is not None else None
}])

(res.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("gold.hypothesis_weekday_weekend"))


In [0]:
# Correlations done correctly (avoid nonsense)
## - Daily AOV vs daily conversion
daily_corr = (
    events.groupBy("event_date")
    .agg(
        F.sum(F.when(F.col("event_type")=="purchase", F.col("price")).otherwise(0)).alias("revenue"),
        F.sum(F.when(F.col("event_type")=="purchase", 1).otherwise(0)).alias("purchases"),
        F.sum(F.when(F.col("event_type")=="view", 1).otherwise(0)).alias("views")
    )
    .withColumn("aov", F.when(F.col("purchases")==0, F.lit(None)).otherwise(F.col("revenue")/F.col("purchases")))
    .withColumn("conversion_rate", F.when(F.col("views")==0, F.lit(None)).otherwise(F.col("purchases")/F.col("views")))
    .select("aov","conversion_rate")
    .na.drop()
)

print("corr(aov, conversion_rate):", daily_corr.stat.corr("aov","conversion_rate"))


In [0]:
# Save the data
(daily_corr.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("gold.daily_aov_vs_conversion"))


In [0]:
# Feature engineering for ML (session/user/product features)
## -Event-level features + label
w_user_time = Window.partitionBy("user_id").orderBy("event_ts")

feat = (
    events
    .withColumn("label", (F.col("event_type")=="purchase").cast("int"))
    .withColumn("price_log", F.log1p(F.coalesce(F.col("price"), F.lit(0.0))))
    .withColumn("prev_event_ts", F.lag("event_ts").over(w_user_time))
    .withColumn("secs_since_prev_event",
                F.when(F.col("prev_event_ts").isNull(), None)
                 .otherwise(F.unix_timestamp("event_ts") - F.unix_timestamp("prev_event_ts")))
)

feat.select("user_id","event_ts","event_type","label","price_log","secs_since_prev_event","is_weekend","event_hour").show(10, truncate=False)


In [0]:
## -Rolling behavioral features (last N events)
w_user_20 = w_user_time.rowsBetween(-20, -1)

feat = (
    feat
    .withColumn("prev_views_20", F.sum(F.when(F.col("event_type")=="view", 1).otherwise(0)).over(w_user_20))
    .withColumn("prev_carts_20", F.sum(F.when(F.col("event_type")=="cart", 1).otherwise(0)).over(w_user_20))
    .withColumn("prev_purchases_20", F.sum(F.when(F.col("event_type")=="purchase", 1).otherwise(0)).over(w_user_20))
)

In [0]:
## - User lifetime aggregates (as-of features)
user_agg = (
    events.groupBy("user_id")
    .agg(
        F.count("*").alias("user_event_cnt"),
        F.sum(F.when(F.col("event_type")=="purchase", 1).otherwise(0)).alias("user_purchase_cnt"),
        F.sum(F.when(F.col("event_type")=="purchase", F.col("price")).otherwise(0)).alias("user_total_spent"),
        F.max("event_ts").alias("user_last_event_ts")
    )
)

feat = feat.join(user_agg, on="user_id", how="left")

In [0]:
## - Product popularity signals (leakage-safe option)
prod_agg = (
    events.groupBy("product_id")
    .agg(
        F.sum(F.when(F.col("event_type")=="view", 1).otherwise(0)).alias("prod_views"),
        F.sum(F.when(F.col("event_type")=="purchase", 1).otherwise(0)).alias("prod_purchases"),
        F.sum(F.when(F.col("event_type")=="purchase", F.col("price")).otherwise(0)).alias("prod_revenue"),
    )
    .withColumn("prod_conversion", F.when(F.col("prod_views")==0, F.lit(0.0))
                .otherwise(F.col("prod_purchases")/F.col("prod_views")))
)

feat = feat.join(prod_agg, on="product_id", how="left")

In [0]:
# Final feature table (select what is required)
feature_cols = [
    "label",
    "user_id",
    "product_id",
    "category_code",
    "brand",
    "event_date",
    "event_hour",
    "is_weekend",
    "price_log",
    "secs_since_prev_event",
    "prev_views_20",
    "prev_carts_20",
    "prev_purchases_20",
    "user_event_cnt",
    "user_purchase_cnt",
    "user_total_spent",
    "prod_views",
    "prod_purchases",
    "prod_revenue",
    "prod_conversion",
]

train = feat.select(*feature_cols)

(train.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("gold.ml_features_purchase_intent"))
